# One-sample video-agent diagnostic pipeline

## Engineering handoff for a new coworker

This notebook explains the repository's smallest trustworthy execution path from raw video to model answer. It is written for someone who has not seen this project before and may later maintain the code without AI assistance.

**Scope:** one Video-MME sample (`question_id = 653-1`) in four deliberately separate modes. This is not an accuracy experiment and not a benchmark report.

**What was intentionally not changed:** the Qwen3-VL model, prompts, answer parser, tool schemas, crop behavior, and FlashVID compression algorithm.

**Safety:** all expensive cells are disabled by default. Reading this notebook and running the inspection cells does not load the model or use a GPU.

## 1. The five-minute summary

The system answers a multiple-choice question about a long video. It begins with 64 uniformly sampled frames. Depending on the mode, it either answers immediately, receives a manually forced crop/compressed view, or autonomously chooses tools.

| Mode | Who chooses visual evidence? | Generations | Tool calls | Observed result |
|---|---:|---:|---:|---|
| `baseline_single_turn` | Harness supplies initial view only | exactly 1 | exactly 0 | Parsed `C`, gold `C` |
| `forced_crop` | Human supplies `0–300s` | exactly 1 | 0 model-selected calls | Model emitted invalid `<answer>Unknown</answer>` |
| `forced_compress` | Human supplies `0–300s` | exactly 1 | 0 model-selected calls | Tensors valid, but 5,895 tokens exceeded budget 5,760 |
| `autonomous_tools` | Model chooses tools | 4 including finalizer | 2 | Protocol worked; parsed `D`, gold `C` |

The important conclusion is not that one mode is more accurate. The important conclusion is that we can now attribute failures to a specific layer:

- baseline control flow is correct and truly single-turn;
- forced crop executed correctly, but the model refused the A–D contract;
- forced compression produced finite, dimensionally valid tensors, but violated the expected hard token budget;
- autonomous tool syntax and execution worked, but the model's final answer was wrong.

In [ ]:
# Lightweight setup: standard library only, no model import and no GPU use.
from pathlib import Path
import json
from pprint import pprint

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "fast_agent" / "debug_pipeline.py").exists():
            return candidate
    raise FileNotFoundError("Could not find longvt_compression repo root")

REPO_ROOT = find_repo_root(Path.cwd())
RUN_ROOT = REPO_ROOT / "debug_runs" / "653-1"
MODES = ["baseline_single_turn", "forced_crop", "forced_compress", "autonomous_tools"]
RUN_MODEL = False  # Deliberately guarded. Change only when an isolated GPU is available.

print("Repository:", REPO_ROOT)
print("Artifacts:", RUN_ROOT)
assert all((RUN_ROOT / mode).is_dir() for mode in MODES)

## 2. The concrete sample

- **Question ID:** `653-1`
- **Video ID:** `YtAL8y2lACs`
- **Video duration:** about 2,839.94 seconds
- **Question:** “How many people does the Ross Ice Shelf team consist of?”
- **Gold option:** `C`
- **Manual forced span:** `0–300s`

Every conclusion below comes from this one sample. Do not generalize accuracy from it.

In [ ]:
def load_json(mode: str, filename: str):
    return json.loads((RUN_ROOT / mode / filename).read_text())

results = {mode: load_json(mode, "parsed_result.json") for mode in MODES}
summary_fields = ["status", "failure_component", "parsed_answer", "gold",
                  "generation_count", "tool_call_count", "stopping_reason"]
for mode, result in results.items():
    print(f"\n[{mode}]")
    pprint({key: result.get(key) for key in summary_fields if key in result})

## 3. Repository map: where to read and where to edit

| File | Responsibility | Change only when... |
|---|---|---|
| `fast_agent/config.py` | Model/data paths, sampling limits, token budgets, prompts, tool schemas | You intentionally change an experiment contract |
| `fast_agent/data.py` | Loads Video-MME rows, formats questions, parses A–D answers | Dataset or answer contract changes |
| `fast_agent/tools.py` | Plans timestamps, decodes frames, creates crop tensors/compression tensors | Sampling or decoding is wrong |
| `fast_agent/model.py` | Loads Qwen3-VL, encodes images, calls FlashVID, creates `Clip` objects | Vision/compression boundary is wrong |
| `fast_agent/agent_loop.py` | Renders chat templates, assembles embeddings/M-RoPE, decodes, legacy autonomous loop | Token layout or general loop behavior is wrong |
| `fast_agent/debug_pipeline.py` | The four isolated diagnostic modes and artifact writer | Diagnostic visibility/control flow is wrong |

Start debugging in `debug_pipeline.py`. Move downward into tools/model/assembly only when the artifacts prove that layer is responsible. Do not start by editing prompts or FlashVID.

In [ ]:
def show_source(relative_path: str, first_line: int, last_line: int):
    """Print exact current source with stable line numbers."""
    path = REPO_ROOT / relative_path
    lines = path.read_text().splitlines()
    for number in range(first_line, min(last_line, len(lines)) + 1):
        print(f"{number:4d}  {lines[number - 1]}")

# The isolated baseline and its three required assertions.
show_source("fast_agent/debug_pipeline.py", 144, 169)

## 4. Why the old baseline sometimes generated twice

The old `run_sample()` loop did stop after the first baseline generation because there were no enabled tools. The extra generation happened **after** that loop:

```python
pred_strict = data.extract_answer("\n".join(texts))
if pred_strict is None:
    messages.append({"role": "user", "parts": ["Reply with <answer>X</answer> ..."]})
    _text, rec = gen("finalizer")
```

Therefore the exact condition responsible was `if pred_strict is None:` in `fast_agent/agent_loop.py`.

The new `baseline_single_turn` does not call `run_sample()` and has no path to a finalizer. Its contract is:

```text
question + initial visual input
              │
              ▼
       exactly one generation
              │
              ▼
        parse A–D, then stop
```

This distinction matters: a parser miss may make the answer unusable, but it must not silently change a single-turn experiment into a two-turn experiment.

In [ ]:
# Inspect the exact legacy finalizer condition in the current source.
show_source("fast_agent/agent_loop.py", 344, 357)

## 5. Shared data path

All four modes share the same low-level route:

```text
MP4 video
  └─ tools.py: sampling plan → exact timestamps → decoded RGB frames
       ├─ image path: list[PIL.Image] → Engine.encode_images()
       └─ video path: uint8 (T,C,H,W) → Engine.encode_video_compressed()
              └─ vision tower → cls attention → FlashVID → kept visual tokens

Clip objects + text messages
  └─ _render_text(): chat template with image/video sentinel placeholders
       └─ assemble(): expand visual pads, compute full-grid M-RoPE,
                     insert embeddings, gather kept FlashVID positions
            └─ decode(): one greedy language-model generation
                 └─ data.extract_answer(): A/B/C/D or None
```

Two key in-memory objects:

- `Clip`: cached visual embeddings, deepstack embeddings, grid metadata, and for compressed video the kept anchor indices.
- `Assembled`: final `(1, L, D)` embeddings, `(3, 1, L)` M-RoPE positions, visual mask, deepstack rows, and kept token IDs.

The vision tower is run once per visual block. Later autonomous rounds reassemble cached clips rather than re-encoding them.

## 6. Mode A — baseline single turn

The baseline sees 64 uniformly sampled frames from the entire video. It performs exactly one generation, parses the result, and stops.

![Initial 64-frame overview](baseline_single_turn/visualizations/initial.png)

**Observed:** the model returned `<answer>C</answer>`, which matched gold `C`. More importantly, all three control-flow assertions passed.

In [ ]:
baseline = results["baseline_single_turn"]
pprint(baseline["assertions"])
print("\nRaw generation:\n")
print((RUN_ROOT / "baseline_single_turn" / "raw_generations.txt").read_text())

## 7. Mode B — forced crop

The harness manually decoded the supplied `0–300s` span. The model was never asked whether it wanted a crop and no tool schema was needed for this operation.

- planned timestamps: 128 values in `frame_timestamps.json`;
- decoded frames: 128;
- resolution: 160×288;
- exact rendered model input: `prompt.txt`;
- model generations: exactly one.

![Forced crop montage](forced_crop/visualizations/forced_crop.png)

**Observed failure:** `<answer>Unknown</answer>`. The parser correctly rejects this because the unchanged contract permits only A–D. This is **model behavior**, not a parser or crop-path bug.

In [ ]:
crop_result = results["forced_crop"]
crop_times = load_json("forced_crop", "frame_timestamps.json")["forced_crop"]
crop_shapes = load_json("forced_crop", "tensor_shapes.json")
print("First timestamps:", crop_times[:8])
print("Last timestamps:", crop_times[-8:])
print("Frame metadata:", crop_shapes["crop_frames"])
print("\nRaw generation:\n", crop_result["raw_output"])
print("\nParsed answer:", crop_result["parsed_answer"])


## 8. Mode C — forced FlashVID compression

This mode uses the same manual `0–300s` span, but decodes it as a video tensor and sends it through the existing Qwen3-VL vision tower plus unchanged FlashVID compressor.

![Frames supplied to the compression path](forced_compress/visualizations/sampled_source_frames.png)

### Exact observed boundary

| Quantity | Shape/value | Meaning |
|---|---:|---|
| decoded input | `(300, 3, 160, 288)` | 300 RGB frames |
| processor patch rows | `(27000, 1536)` | flattened spatiotemporal patch inputs |
| vision hidden | `(6750, 4096)` | merged visual tokens before FlashVID |
| `video_features` | `(150, 45, 4096)` | 150 temporal groups × 45 spatial tokens |
| CLS attention | `(150, 45)` | selection signal consumed by FlashVID |
| each pre-FlashVID deepstack | `(6750, 4096)` | three visual feature levels |
| tokens before → after | `6750 → 5895` | actual compression result |
| expected budget | `5760` | 128 crop frames × 45 tokens/frame |
| assembled embeddings | `(1, 10513, 4096)` | text + initial visuals + compressed visuals |
| assembled M-RoPE | `(3, 1, 10513)` | temporal/height/width position planes |

All finite-value, dimensionality, and nonzero-token assertions passed. The hard budget assertion failed because 5,895 is 135 tokens (2.34%) over 5,760.

Why this can happen: the configured retention ratio is an input to FlashVID, not an exact postcondition. Segment-floor/anchor behavior can retain more positions than the nominal ratio predicts. The diagnostic harness intentionally reports this rather than silently relaxing the expected budget.

In [ ]:
compress_result = results["forced_compress"]
compress_shapes = load_json("forced_compress", "tensor_shapes.json")
print("Assertions:")
pprint(compress_result["assertions"])
print("\nToken accounting:")
pprint(compress_shapes["flashvid_tokens"])
print("\nFlashVID boundary shapes and finite checks:")
pprint(compress_shapes["flashvid_boundary"])
print("\nFinal assembly shapes:")
pprint(compress_shapes["generation_1"])
print("\nExact surrounding text at video insertion:")
pprint(compress_shapes["inserted_visual_text_context"])


### Where compression happens in code

`Engine.encode_video_compressed()` performs these steps:

1. The video processor converts `(T,C,H,W)` uint8 frames into patch rows and `video_grid_thw`.
2. The patched vision tower returns `hidden`, `deepstack`, and last-block `cls_attention`.
3. `hidden` is reshaped to `(temporal_groups, spatial_tokens, hidden_dim)`.
4. `flashvid_compression(video_features, cls_attention, flashvid_config)` returns compressed features and `keep_idx`.
5. Deepstack rows are gathered with the same indices.
6. Native timestamp text plus full video-pad segments are built.
7. During assembly, M-RoPE is computed on the **full grid first**, then non-kept video positions are gathered. Duplicate anchors remain duplicates to match FlashVID semantics.

Do not move compression into `tools.py`: the real compression boundary requires vision embeddings and CLS attention.

In [ ]:
# Exact current instrumentation around the unchanged FlashVID call.
show_source("fast_agent/model.py", 243, 303)

## 9. Mode D — autonomous tools

This mode is deliberately separate from both forced modes. The model receives native tool schemas and decides whether and where to inspect.

### Actual timeline

1. **Generation 1:** called `compress_video(0, 2840)`.
   - clamped end: 2,839.9371s
   - decoded 2,128 frames at 160×288
   - compressed 47,880 → 5,670 visual tokens
2. **Generation 2:** called `crop_video(183, 188)`.
   - decoded five frames at 160×288
3. **Generation 3:** emitted plain text: `The Ross Ice Shelf team consists of 8 people.`
   - no `<answer>A-D</answer>` and no executable tool call, so the normal loop stopped
4. **Finalizer:** emitted `<answer>D</answer>`.
   - parser returned `D`; gold was `C`

The tool protocol worked. The wrong answer is model behavior, not evidence that tool parsing or execution failed.

### Autonomous wide compression

![Autonomous full-video compressed overview](autonomous_tools/visualizations/tool_1_compress.png)

### Autonomous localized crop, 183–188s

![Autonomous five-frame crop](autonomous_tools/visualizations/tool_2_crop.png)

In [ ]:
auto = results["autonomous_tools"]
for event in auto["events"]:
    print("\n" + "=" * 72)
    print("Generation:", event["generation"])
    print("Raw output:", event["raw_output"])
    print("Parsed tool call:", event.get("parsed_tool_call"))
    print("Tool result:", event.get("tool_result"))
    print("Stopping reason:", event.get("stopping_reason"))
    print("Parsed answer:", event.get("parsed_answer"))

## 10. Artifact contract

Every mode directory must contain:

- `config.json` — model/data/budget/span configuration;
- `prompt.txt` — exact rendered input for every generation;
- `raw_generations.txt` — unmodified model text;
- `trace.md` — step-by-step plain-English account;
- `parsed_result.json` — structured outcome and failure ownership;
- `frame_timestamps.json` — exact planned timestamps for every decoded visual block;
- `tensor_shapes.json` — frame, visual, FlashVID, assembly, and M-RoPE diagnostics;
- `visualizations/` — initial views, crop montages, or compression source montages.

`parsed_result.json` is the machine-facing verdict. `trace.md` is the first file a human should open. `prompt.txt` and `raw_generations.txt` are the source of truth when diagnosing model behavior.

In [ ]:
required = {
    "config.json", "prompt.txt", "raw_generations.txt", "trace.md",
    "parsed_result.json", "frame_timestamps.json", "tensor_shapes.json",
}
for mode in MODES:
    mode_dir = RUN_ROOT / mode
    present = {path.name for path in mode_dir.iterdir() if path.is_file()}
    missing = sorted(required - present)
    visuals = sorted(path.name for path in (mode_dir / "visualizations").glob("*"))
    print(f"{mode:22s} missing={missing} visuals={visuals}")
    assert not missing

## 11. How to diagnose a future failure

Use this order. It prevents fixing the wrong layer.

1. **Harness:** Did the requested mode run? Are generation/tool counts correct? Did all required files get written?
2. **Decode/crop path:** Do timestamps, frame count, resolution, and montage match the requested span?
3. **Compression path:** Are tensors finite and dimensions valid? Are before/after tokens and M-RoPE shapes sensible? Did the budget assertion pass?
4. **Tool protocol:** Does raw output contain a valid tool JSON block? Was it clamped, executed, and returned to the model exactly once?
5. **Parser:** Does raw output satisfy the unchanged A–D contract? Never call `<answer>Unknown</answer>` a parser bug.
6. **Model behavior:** Only after the five mechanical layers pass should you discuss evidence choice or answer quality.

A correct pipeline can produce a wrong answer. A correct answer can also hide a broken experimental contract. Record both separately.

## 12. Smallest next engineering tasks

Do these independently; do not combine them into an architecture refactor.

### A. Recheck forced crop localization

The autonomous mode localized `183–188s`. Rerun only `forced_crop` on that manual span. This tests whether the original `0–300s` crop was simply too broad. It requires no prompt/parser/tool change.

### B. Decide what “token budget” means

The current harness treats 5,760 as a hard maximum and correctly fails at 5,895. Before changing compression, write down one of two contracts:

- **hard cap:** post-compression token count must be `<= target`; or
- **nominal target:** segment-floor overhead is permitted within an explicitly documented tolerance.

If hard cap is chosen, add a focused test before changing FlashVID selection/pruning. That would be an algorithm change and is intentionally not done here.

### C. Preserve autonomous observability

The model wrote an untagged numeric answer, causing the existing finalizer. Do not “fix” this by weakening the parser until you first decide whether the experiment requires strict tagged answers or accepts free-form numeric reasoning.

### D. Add small CPU/mocked tests

- baseline always records one generation and zero tools;
- forced modes never parse or execute a model-selected tool call;
- artifact writer produces all required files on both success and failure;
- answer-contract ownership classifies `<answer>Unknown</answer>` as model behavior;
- compression result preserves detailed assertion values even when budget fails.

## 13. Guarded commands

The working interpreter is:

```text
/local1/cfyang/miniconda3/envs/flashvid/bin/python
```

`fast_agent` is local source, not an installed package. Run commands from `/home/cfyang/hanklin/longvt_compression` or prepend that directory to `PYTHONPATH`.

Run exactly one mode with `--mode`. Never use `run_eval.py` for this diagnostic workflow.

```bash
CUDA_VISIBLE_DEVICES=<free_gpu> \
/local1/cfyang/miniconda3/envs/flashvid/bin/python -u \
-m fast_agent.debug_pipeline \
--qid 653-1 --start 183 --end 188 \
--output-root debug_runs --mode forced_crop
```

Before running, check both `nvidia-smi` and active processes. A previous attempt hit OOM after another process occupied the initially free GPU.

In [ ]:
# Heavy execution remains opt-in. This cell is safe as committed.
if RUN_MODEL:
    raise RuntimeError(
        "Do not launch from the notebook blindly. First choose a free GPU, then "
        "run the explicit shell command in the previous cell so GPU ownership "
        "and the selected mode are visible in command history."
    )
else:
    print("RUN_MODEL=False: no model loaded, no video decoded, no GPU used.")

## 14. Definition of done for future changes

Before claiming a diagnostic change works:

- run `python -m py_compile fast_agent/debug_pipeline.py fast_agent/tools.py fast_agent/model.py`;
- run mocked/unit checks for control flow without a GPU;
- run only the affected mode on exactly one sample;
- validate every JSON artifact;
- inspect the saved montage rather than trusting frame counts alone;
- confirm no diagnostic/model process remains after completion;
- report pipeline success separately from answer correctness;
- do not launch the benchmark.

The current artifact set meets this handoff contract. The open technical issue is the compression hard-budget violation; the open behavioral issues are invalid/incorrect model answers. Those are separate problems and should remain separate in code and reporting.

# Appendix: exact source-code snapshots

The cells below embed the **actual complete source files** used by this diagnostic run. They are not summaries and do not read code from disk when viewed. This makes the notebook self-contained for a coworker or another LLM acting as a tutor.

Read them in this order:

1. `config.py` — contracts and constants;
2. `data.py` — sample loading and answer parsing;
3. `tools.py` — frame planning/decoding;
4. `model.py` — image/video encoding and FlashVID;
5. `agent_loop.py` — message rendering, token assembly, M-RoPE, decoding, legacy loop;
6. `trajectory.py` — visualization and trajectory persistence;
7. `debug_pipeline.py` — the four diagnostic modes and artifact contract.

Each heading includes a SHA-256 digest so future maintainers can detect when the live file has drifted from this handoff snapshot. These are Markdown code blocks, so opening or running the notebook cannot execute them.


## Source snapshot: `fast_agent/config.py`

- Lines: `163`
- SHA-256: `1fbb61ffeb80e4094747172dd9aa15431b22ff08983a6a06697ebc0d45838567`

```python
"""fast_agent configuration — paths, budgets, prompts.

Design decisions (see ~/.claude/plans/ok-fix-the-prompt-magical-globe.md):
- zero-shot Qwen3-VL-8B-Instruct, local HF, flash-attn-2, batch=1 per worker
- LongVT-faithful loop: native tool schema (# Tools block), terse prompt, single-turn
  baseline, one bounded finalizer, dual (strict/lenient) scoring
- initial view: 64 frames as IMAGES (deliberately sparse; compression gets room to shine)
- crop_video: <=128 frames, fps=1, 224px, images (full detail)
- compress_video: video modality -> faithful FlashVID (DySeg+ADTS+TSTM), with
  fixed retention and a short-span floor so FINAL token count stays near the
  crop budget (matched-budget A/B)
"""

import os

# ---- paths ----
MODEL_SNAPSHOT = (
    "/local1/cfyang/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/"
    "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b"
)
FLASHVID_REPO = "/home/cfyang/hanklin/FlashVID"
VIDEOMME_PARQUET = (
    "/local1/cfyang/.cache/huggingface/hub/datasets--lmms-lab--Video-MME/snapshots/"
    "ead1408f75b618502df9a1d8e0950166bf0a2a0b/videomme/test-00000-of-00001.parquet"
)
VIDEO_DIR = "/local1/cfyang/.cache/huggingface/videomme/videomme/data"
OUTPUT_DIR = "/local1/cfyang/hanklin/outputs/fast_agent"

# ---- sampling (paper-faithful base settings: fps=1, 224px) ----
FPS = 1
MAX_PIXELS = 224 * 224          # per-frame pixel budget (both tools + initial view)
MIN_PIXELS = 28 * 28
INITIAL_FRAMES = 64             # deliberately sparse initial view (user decision)
CROP_MAX_FRAMES = 128           # crop tool: full-detail budget ceiling
TEMPORAL_PATCH_SIZE = 2         # Qwen3-VL merges this many raw frames per grid-t step

# ---- FlashVID ----
# retention is FIXED (not auto-derived). To keep compress_video matched-budget
# against crop_video (same final token cost, more temporal coverage) rather
# than just cheaper, COMPRESS_MAX_FRAMES must scale inversely with retention.
# Naive derivation: temporal merge already halves raw-frame cost for free (2 raw
# frames -> 1 grid-t slice of P tokens, same P crop_video pays per single
# frame), so matching crop's P*CROP_MAX_FRAMES budget needs
#   N = CROP_MAX_FRAMES * TEMPORAL_PATCH_SIZE / retention
# and this part IS video-independent (P cancels). BUT FlashVID's actual
# kept/base ratio drifts above the nominal retention_ratio (segment-floor
# effect: min_segment_num=8 segments each keep a fixed ADTS token allocation
# regardless of span, so a fixed per-segment cost eats a bigger share of a
# smaller nominal budget) -- and that drift is NOT a fixed percentage, it gets
# worse as retention drops. Measured on a real 3169s video, sweeping N up to
# 3072 with the resolution-ceiling fix in place:
#   retention=0.2: naive N=1280, real crossover ~1224 frames (~5% over)
#   retention=0.1: naive N=2560, real crossover ~2127 frames (~20% over)
# So the naive formula is only a starting point -- use it to size a
# calibration sweep, then read the real crossover off measured kept_tokens.
# Calibrated values (recalibrate if retention changes):
_CALIBRATED_MAX_FRAMES = {0.2: 1224, 0.1: 2128}
FIXED_RETENTION = float(os.environ.get("FA_FIXED_RETENTION", "0.1"))
COMPRESS_MAX_FRAMES = (
    _CALIBRATED_MAX_FRAMES.get(FIXED_RETENTION)
    or (int(round(CROP_MAX_FRAMES * TEMPORAL_PATCH_SIZE / FIXED_RETENTION))
        if FIXED_RETENTION > 0 else 768)
)
# Below the calibrated input-frame cap, fixed retention alone would undershoot
# the crop budget; use the nominal matched-budget ratio for those short spans.
FLOOR_ENGAGE_FRAMES = COMPRESS_MAX_FRAMES
FLASHVID_KW = dict(
    alpha=0.7,                   # faithful default: selection + TSTM merge
    do_segment=True,
    segment_threshold=0.9,
    min_segment_num=8,
    complementary_segment=True,
    token_selection_method="attn_div_v2",
    temporal_threshold=0.8,
    expansion=1.0,               # keep budget math exact (walkthrough default 1.25)
)

# ---- agent loop ----
MAX_ROUNDS = 5                   # tool rounds before tools are withheld
MAX_NEW_TOKENS = 2048            # per round (1024 truncated frame-by-frame analyses
                                 # mid-thought -> deterministic restart loops)
GEN_TEMPERATURE = 0.0            # greedy

# ---- outputs ----
RUN_ROOT = os.path.join(OUTPUT_DIR, "runs")  # per-run dirs: results.jsonl + traj/*.json + media/

# ---- prompts (LongVT-faithful: terse; tool signatures come from the native # Tools block) ----
ANSWER_INSTR = (
    "Think first inside <think></think> tags. If you need to inspect the video more "
    "closely, call one tool; otherwise give your final answer as <answer>X</answer> "
    "where X is one of the option letters A, B, C, or D."
)

# Tombstone: REASONED mode (FA_REASONED) removed 2026-07. Its per-turn "write 2-5
# sentences" instruction caused frame-by-frame narration -> greedy repetition ->
# multi-round refusal loops. Kept as "" so existing imports don't break.
TOOL_RESULT_INSTR = ""

# OpenAI-format tool schemas fed to the model's native chat template (`tools=`), so a
# zero-shot Qwen3-VL sees the `# Tools` block hermes was trained on. The loop owns the
# video, so tools take only start_time/end_time (no video_path).
TOOL_SCHEMAS = {
    "crop_video": {
        "type": "function",
        "function": {
            "name": "crop_video",
            "description": (
                "Zoom in on a specific time span and view it at FULL detail (1 frame per "
                f"second, up to {CROP_MAX_FRAMES} frames). Use a TIGHT span (<=120 seconds) "
                "to read fine details, on-screen text, or faces."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "start_time": {"type": "number", "description": "Start time in seconds."},
                    "end_time": {"type": "number", "description": "End time in seconds, must be greater than start_time."},
                },
                "required": ["start_time", "end_time"],
            },
        },
    },
    "compress_video": {
        "type": "function",
        "function": {
            "name": "compress_video",
            "description": (
                "View a COMPRESSED overview of a LONG span (up to the whole video): many "
                "more frames at the same token cost as crop_video, trading per-frame detail "
                "for temporal coverage. Use it to LOCATE where the answer is across a wide "
                "span, then crop_video to zoom in on the exact moment."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "start_time": {"type": "number", "description": "Start time in seconds."},
                    "end_time": {"type": "number", "description": "End time in seconds, must be greater than start_time."},
                },
                "required": ["start_time", "end_time"],
            },
        },
    },
}

def initial_view_text(duration: float, n_frames: int) -> str:
    return (
        f"This video is {duration:.0f} seconds long. The {n_frames} frames above are "
        f"uniformly sampled from 0s to {duration:.0f}s."
    )

def tool_instructions(duration: float, tools: tuple) -> str:
    """Short strategic hint in the user turn. Tool signatures + call format come from
    the native # Tools block (TOOL_SCHEMAS); this only nudges WHEN to use each."""
    if not tools:
        return ""
    hints = ["\nYou may inspect the video before answering."]
    if "compress_video" in tools and "crop_video" in tools:
        hints.append(
            "Strategy: use compress_video to skim a wide span and locate the relevant "
            "moment, then crop_video to zoom in at full detail. One tool call per turn."
        )
    elif "crop_video" in tools:
        hints.append("Use crop_video to zoom in on the relevant span. One tool call per turn.")
    return " ".join(hints)
```


## Source snapshot: `fast_agent/data.py`

- Lines: `63`
- SHA-256: `a0dd4a3cb4d742c9b4423c53a7ed4d7c33881533c36a324c7c715c1100080f67`

```python
"""VideoMME long-split loading + scoring helpers."""

import os
import re

import cv2
import pandas as pd

from . import config


def load_long_split(n: int | None = None, seed: int = 0) -> list[dict]:
    """Rows of the long split whose video exists locally. Stable order; optional
    stratified-ish head sample (shuffle with fixed seed, then take n)."""
    df = pd.read_parquet(config.VIDEOMME_PARQUET)
    df = df[df["duration"] == "long"].copy()
    have = {f[:-4] for f in os.listdir(config.VIDEO_DIR) if f.endswith(".mp4")}
    df = df[df["videoID"].isin(have)]
    if n is not None:
        df = df.sample(frac=1.0, random_state=seed).head(n)
    rows = []
    for _, r in df.iterrows():
        rows.append(
            {
                "question_id": r["question_id"],
                "videoID": r["videoID"],
                "video_path": os.path.join(config.VIDEO_DIR, r["videoID"] + ".mp4"),
                "question": r["question"],
                "options": list(r["options"]),
                "answer": r["answer"],
                "task_type": r["task_type"],
            }
        )
    return rows


def video_duration(path: str) -> float:
    cap = cv2.VideoCapture(path)
    try:
        fps = cap.get(cv2.CAP_PROP_FPS)
        n = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        return n / fps if fps > 0 else 0.0
    finally:
        cap.release()


def format_question(row: dict) -> str:
    opts = "\n".join(row["options"])
    return f"{row['question']}\n{opts}"


_ANS_RE = re.compile(r"<answer>\s*\(?([A-D])\)?", re.IGNORECASE)
_FALLBACK_RE = re.compile(r"\b([A-D])\b")


def extract_answer(text: str) -> str | None:
    m = _ANS_RE.search(text)
    if m:
        return m.group(1).upper()
    # fallback: last bare option letter in the final line(s)
    tail = text.strip()[-200:]
    hits = _FALLBACK_RE.findall(tail)
    return hits[-1].upper() if hits else None
```


## Source snapshot: `fast_agent/tools.py`

- Lines: `248`
- SHA-256: `c1c141be6762a3ee3b5babd66ebad6c81868acbbd1d859e197ddf3da48a4277f`

```python
"""Video span decoding for the two tools + the initial view.

cv2-based decode (qwen_vl_utils' torchcodec/torchvision backends are broken on
this box — torchcodec can't identify the stream, torchvision/av dies in
swscaler). Sampling matches the project lineage: fps=1 over the span, capped
per-tool, uniform frame centers. Frames are smart-resized (Qwen formula: dims
multiple of 32, area <= MAX_PIXELS) so the processor's own resize is a no-op.
"""

import math
import os
import subprocess
import time

import cv2
import numpy as np
import torch
from PIL import Image

from . import config

PROXY_DIR = os.path.join(config.OUTPUT_DIR, "proxies")


def _smart_size(h: int, w: int, factor: int = 32, max_pixels: int = config.MAX_PIXELS):
    """Qwen smart_resize: round dims to multiples of `factor`, keep area <= max_pixels."""
    if h * w > max_pixels:
        beta = math.sqrt(h * w / max_pixels)
        h, w = h / beta, w / beta
    h = max(factor, round(h / factor) * factor)
    w = max(factor, round(w / factor) * factor)
    while h * w > max_pixels:
        h, w = h - factor if h >= w else h, w - factor if w > h else w
    return int(h), int(w)


_BACKEND_CACHE: dict[str, str] = {}   # original path -> decodable path (self or proxy)


def _plan(video_path: str, start, end, max_frames: int, even: bool):
    """Common sampling plan: clamped span, target frame times, target size."""
    cap = cv2.VideoCapture(video_path)
    try:
        fps_v = cap.get(cv2.CAP_PROP_FPS) or 30.0
        n_v = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        h0 = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        w0 = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    finally:
        cap.release()
    dur = n_v / fps_v if fps_v > 0 else 0.0
    start = 0.0 if start is None else max(0.0, min(start, dur))
    end = dur if end is None else max(start, min(end, dur))
    span = max(end - start, 1e-6)
    n = int(round(span * config.FPS))
    n = max(2, min(n, max_frames))
    if even:
        n = max(2, n - (n % 2))
    centers = start + (np.arange(n) + 0.5) * span / n
    th, tw = _smart_size(h0 or 224, w0 or 224)
    return centers, fps_v, n_v, th, tw


def _decode_cv2(video_path, centers, fps_v, n_v, th, tw):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"cannot open {video_path}")
    try:
        idxs = np.clip((centers * fps_v).astype(int), 0, max(n_v - 1, 0))
        frames = []
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, fr = cap.read()
            if not ok:
                continue
            fr = cv2.resize(fr, (tw, th), interpolation=cv2.INTER_AREA)
            frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        return frames
    finally:
        cap.release()


def _decode_pyav(video_path, centers, th, tw):
    """Software decode via PyAV (handles AV1 through libdav1d; cv2's ffmpeg only
    tries hardware AV1 on this box)."""
    import av

    frames = []
    with av.open(video_path) as container:
        stream = container.streams.video[0]
        tb = stream.time_base
        for t in centers:
            container.seek(int(t / tb), stream=stream)  # keyframe at/before t
            got = None
            for frame in container.decode(stream):
                if frame.time is None or frame.time >= t - 1e-3:
                    got = frame
                    break
            if got is None:
                continue
            arr = got.to_ndarray(format="rgb24")
            frames.append(cv2.resize(arr, (tw, th), interpolation=cv2.INTER_AREA))
    return frames


def make_proxy(video_path: str) -> str:
    """One-time low-res h264 proxy for videos cv2 can't decode (AV1 — this box's
    cv2/ffmpeg lacks software AV1; system ffmpeg has libdav1d). 256px short side
    is lossless for us: every consumer resizes to <=224x224 anyway. Seek-per-frame
    pyav on hour-long sparse-keyframe AV1 measured ~10 min per 48 frames — the
    proxy makes all later decodes cv2-fast."""
    os.makedirs(PROXY_DIR, exist_ok=True)
    proxy = os.path.join(PROXY_DIR, os.path.basename(video_path))
    if os.path.exists(proxy) and os.path.getsize(proxy) > 0:
        return proxy

    # Race-safe: video-sharding already keeps one video on one worker, but guard
    # anyway. O_EXCL lock claims the transcode; other workers wait for the result.
    lock = proxy + ".lock"
    try:
        fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        os.close(fd)
    except FileExistsError:
        for _ in range(900):                       # someone else is transcoding
            if os.path.exists(proxy) and os.path.getsize(proxy) > 0:
                return proxy
            time.sleep(1)
        # stale lock (dead worker) -> fall through and transcode ourselves

    tmp = proxy + f".{os.getpid()}.tmp.mp4"         # unique per process
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-i", video_path,
             "-vf", "scale=-2:256", "-c:v", "libx264", "-preset", "veryfast",
             "-crf", "28", "-an", tmp],
            check=True, timeout=1800,
        )
        os.replace(tmp, proxy)                       # atomic
    finally:
        for f in (lock, tmp):
            try:
                os.remove(f)
            except OSError:
                pass
    return proxy


def resolve_decodable(video_path: str) -> str:
    """Return a cv2-decodable path for this video (original, or cached proxy —
    transcoding once if needed)."""
    cached = _BACKEND_CACHE.get(video_path)
    if cached is not None:
        return cached
    cap = cv2.VideoCapture(video_path)
    try:
        n_v = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(n_v // 2 - 1, 0))
        ok, _ = cap.read()
    finally:
        cap.release()
    path = video_path if ok else make_proxy(video_path)
    _BACKEND_CACHE[video_path] = path
    return path


def _decode_span_with_timestamps(
    video_path: str,
    start: float | None,
    end: float | None,
    max_frames: int,
    even: bool = False,
) -> tuple[list[np.ndarray], list[float]]:
    """Decode a span and retain the sampling-plan timestamps for diagnostics."""
    src = resolve_decodable(video_path)
    centers, fps_v, n_v, th, tw = _plan(src, start, end, max_frames, even)
    frames = _decode_cv2(src, centers, fps_v, n_v, th, tw)
    if len(frames) < 2:  # last resort: software decode of the original
        frames = _decode_pyav(video_path, centers, th, tw)
    if len(frames) < 2:
        raise RuntimeError(f"decoded {len(frames)} frames from {video_path}")
    if even and len(frames) % 2:
        frames = frames[:-1]
    return frames, centers[: len(frames)].astype(float).tolist()


def _decode_span(video_path: str, start: float | None, end: float | None,
                 max_frames: int, even: bool = False) -> list[np.ndarray]:
    frames, _timestamps = _decode_span_with_timestamps(
        video_path, start, end, max_frames, even
    )
    return frames


def initial_frames(video_path: str) -> list:
    """48 uniformly-sampled PIL frames of the whole video (image modality)."""
    return [Image.fromarray(f) for f in
            _decode_span(video_path, None, None, config.INITIAL_FRAMES)]


def initial_frames_with_timestamps(video_path: str):
    """Diagnostic initial-view API with the exact planned sample timestamps."""
    frames, timestamps = _decode_span_with_timestamps(
        video_path, None, None, config.INITIAL_FRAMES
    )
    return [Image.fromarray(f) for f in frames], timestamps


def crop_frames(video_path: str, start: float, end: float) -> list:
    """<=128 PIL frames at fps=1 over [start, end] (full-detail image modality)."""
    return [Image.fromarray(f) for f in
            _decode_span(video_path, start, end, config.CROP_MAX_FRAMES)]


def crop_frames_with_timestamps(video_path: str, start: float, end: float):
    """Diagnostic crop API; production callers keep using ``crop_frames``."""
    frames, timestamps = _decode_span_with_timestamps(
        video_path, start, end, config.CROP_MAX_FRAMES
    )
    return [Image.fromarray(f) for f in frames], timestamps


def compress_tensor(video_path: str, start: float, end: float):
    """((T,C,H,W) uint8 tensor, frame_times) over [start, end], up to 768 frames,
    T even (video modality for the FlashVID path). frame_times are the sampled
    frames' timestamps in seconds — needed for Qwen3-VL's per-frame timestamp
    text (`<{t:.1f} seconds>` before each temporal group)."""
    frames, times = _decode_span_with_timestamps(
        video_path, start, end, config.COMPRESS_MAX_FRAMES, even=True
    )
    return (
        torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2).contiguous(),
        times,
    )


def clamp_span(start, end, duration: float) -> tuple[float, float, str | None]:
    """Sanitize a model-emitted span. Returns (start, end, error_or_None)."""
    try:
        start, end = float(start), float(end)
    except (TypeError, ValueError):
        return 0.0, 0.0, "start_time/end_time must be numbers (seconds)."
    start = max(0.0, min(start, duration))
    end = max(0.0, min(end, duration))
    if end - start < 1.0:
        return start, end, (
            f"Invalid span [{start:.0f}, {end:.0f}]s: end must exceed start by >=1s "
            f"within [0, {duration:.0f}]s."
        )
    return start, end, None
```


## Source snapshot: `fast_agent/model.py`

- Lines: `312`
- SHA-256: `c4d4cafcc93de33193cb7b9624d7bcf57474669831c5117c52e3a739e0189c5e`

```python
"""Engine: Qwen3-VL-8B + vision-side FlashVID patches + VisionService.

Design (verified against FlashVID/flashvid/modeling_qwen3_vl.py):
- We patch ONLY the vision tower (attention/block/model forwards) so `visual()`
  returns (merged_tokens, deepstack_list, cls_attention). The text model stays
  100% stock — no fastv, no flashvid_config plumbing on the LM side.
- The patched vision attention asserts flash_attention_2 → we load with it.
- Compression calls flashvid.utils.flashvid_compression DIRECTLY (same call the
  patched Qwen3VLModel_forward makes at line 307), so DySeg+ADTS+TSTM and real
  last-block CLS attention are exactly FlashVID-faithful.
- Each clip is ViT-encoded ONCE and cached (embeds + deepstack + token segment);
  the agent loop re-assembles embeddings across rounds without re-encoding.
"""

import dataclasses
import sys
from dataclasses import dataclass, field

import torch

from . import config

sys.path.insert(0, config.FLASHVID_REPO)

from flashvid.configuration_flashvid import FlashVidConfig  # noqa: E402
from flashvid.utils import flashvid_compression  # noqa: E402


# ---------------------------------------------------------------------------
@dataclass
class Clip:
    """One encoded visual block, ready to splice into any round's sequence."""

    kind: str                       # "images" | "video"
    segment_ids: torch.Tensor       # (S,) FULL token segment incl. vision_start/end
    embeds: torch.Tensor            # (K, D) tokens to scatter (post-compression for video)
    deepstack: list                 # per-level (K, D)
    grid_thw: torch.Tensor          # (n, 3) grids for get_rope_index (n images or 1 video)
    keep_indices: torch.Tensor | None = None  # video only: kept anchors into full pad run
    meta: dict = field(default_factory=dict)

    @property
    def n_tokens(self) -> int:
        return self.embeds.shape[0]


def clip_tokens(grid_thw, merge_size: int = 2, retention: float = 1.0) -> int:
    """Exact merged-token count for a grid (derive, don't hardcode: patch_size=16)."""
    total = 0
    for t, h, w in grid_thw.tolist():
        total += int(t) * (int(h) // merge_size) * (int(w) // merge_size)
    return int(round(total * retention))


def retention_for_budget(base_tokens: int, target_tokens: int, r_min: float = 0.02) -> float:
    return max(r_min, min(1.0, target_tokens / max(base_tokens, 1)))


# ---------------------------------------------------------------------------
class Engine:
    def __init__(self, device: str = "cuda:0"):
        from transformers import AutoProcessor, AutoTokenizer
        from transformers.models.qwen3_vl.modeling_qwen3_vl import (
            Qwen3VLForConditionalGeneration,
        )

        self.device = device
        self.model = Qwen3VLForConditionalGeneration.from_pretrained(
            config.MODEL_SNAPSHOT,
            torch_dtype=torch.bfloat16,
            attn_implementation="flash_attention_2",
        ).to(device).eval()
        self.processor = AutoProcessor.from_pretrained(
            config.MODEL_SNAPSHOT,
            max_pixels=config.MAX_PIXELS,
            min_pixels=config.MIN_PIXELS,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(config.MODEL_SNAPSHOT)

        cfg = self.model.config
        self.image_token_id = cfg.image_token_id
        self.video_token_id = cfg.video_token_id
        self.vision_start_id = cfg.vision_start_token_id
        self.vision_end_id = cfg.vision_end_token_id
        self.eos_ids = {self.tokenizer.eos_token_id}
        im_end = self.tokenizer.convert_tokens_to_ids("<|im_end|>")
        if im_end is not None:
            self.eos_ids.add(im_end)

        self._vision_patched = False
        self._orig_forwards = {}
        self._fast_patch_embed()

    def _fast_patch_embed(self):
        """Replace patch_embed's Conv3d with the mathematically exact matmul.
        kernel==stride makes the conv a linear map over flattened patches; cuDNN
        picks a catastrophic kernel for this shape on the A6000 (34.3s for 64
        frames vs 0.005s as a matmul — 6900x; max abs diff 0.016 = bf16 noise).
        Instance-level override; all arms share it, so A/Bs stay fair."""
        pe = self.model.model.visual.patch_embed
        w = pe.proj.weight.data
        b = pe.proj.bias.data if pe.proj.bias is not None else None
        w2 = w.view(w.shape[0], -1)

        def fast_forward(hidden_states):
            out = hidden_states.view(hidden_states.shape[0], -1) @ w2.T
            return out + b if b is not None else out

        pe.forward = fast_forward

    # -- vision-side FlashVID patch (reversible) ----------------------------
    def apply_vision_patches(self):
        if self._vision_patched:
            return
        from transformers.models.qwen3_vl.modeling_qwen3_vl import (
            Qwen3VLVisionAttention,
            Qwen3VLVisionBlock,
            Qwen3VLVisionModel,
        )
        from flashvid.modeling_qwen3_vl import (
            Qwen3VLVisionAttention_forward,
            Qwen3VLVisionBlock_forward,
            Qwen3VLVisionModel_forward,
        )

        self._orig_forwards = {
            "attn": Qwen3VLVisionAttention.forward,
            "block": Qwen3VLVisionBlock.forward,
            "model": Qwen3VLVisionModel.forward,
        }
        Qwen3VLVisionAttention.forward = Qwen3VLVisionAttention_forward
        Qwen3VLVisionBlock.forward = Qwen3VLVisionBlock_forward
        Qwen3VLVisionModel.forward = Qwen3VLVisionModel_forward
        self._vision_patched = True

    def restore_vision(self):
        if not self._vision_patched:
            return
        from transformers.models.qwen3_vl.modeling_qwen3_vl import (
            Qwen3VLVisionAttention,
            Qwen3VLVisionBlock,
            Qwen3VLVisionModel,
        )

        Qwen3VLVisionAttention.forward = self._orig_forwards["attn"]
        Qwen3VLVisionBlock.forward = self._orig_forwards["block"]
        Qwen3VLVisionModel.forward = self._orig_forwards["model"]
        self._vision_patched = False

    # -- encoding ------------------------------------------------------------
    @torch.inference_mode()
    def encode_images(self, pils: list, meta: dict | None = None) -> Clip:
        """Encode PIL frames as IMAGE modality (full detail; never compressed).
        ViT runs exactly once; result cached in the returned Clip."""
        self.apply_vision_patches()
        proc = self.processor.image_processor(images=pils, return_tensors="pt")
        pixel_values = proc["pixel_values"].to(self.device, torch.bfloat16)
        grid_thw = proc["image_grid_thw"].to(self.device)
        hidden, deepstack, _attn = self.model.model.visual(pixel_values, grid_thw)

        # segment ids: per image <|vision_start|> pads <|vision_end|>
        seg = []
        for t, h, w in grid_thw.tolist():
            n = t * (h // 2) * (w // 2)
            seg += [self.vision_start_id] + [self.image_token_id] * n + [self.vision_end_id]
        return Clip(
            kind="images",
            segment_ids=torch.tensor(seg, dtype=torch.long, device=self.device),
            embeds=hidden,
            deepstack=list(deepstack),
            grid_thw=grid_thw,
            meta=meta or {},
        )

    @torch.inference_mode()
    def encode_video_compressed(
        self,
        video_tensor: torch.Tensor,
        target_tokens: int,
        frame_times: list[float],
        meta: dict | None = None,
    ) -> Clip:
        """Encode a (T,C,H,W) uint8 clip as VIDEO modality and FlashVID-compress
        to ~target_tokens. Faithful path: patched ViT last-block CLS attention →
        DySeg + ADTS + TSTM merge (alpha=0.7). ViT runs exactly once."""
        import time

        self.apply_vision_patches()
        # do_sample_frames=False: the processor's default (True, fps=2) would
        # RESAMPLE our already-sampled frames and break frame_times alignment.
        #
        # size override: Qwen3VLVideoProcessor has its OWN global token ceiling
        # (video_preprocessor_config.json's size.longest_edge, 25,165,824 for this
        # checkpoint) checked as t_bar*h_bar*w_bar > longest_edge -- a budget on
        # the WHOLE clip's frames*h*w, independent of our own per-frame MAX_PIXELS
        # in tools.py. Once a clip's frame count crosses ~560 (at our 224px-derived
        # per-frame size) it silently downscales resolution to claw back under that
        # ceiling -- confirmed empirically: 512 frames -> untouched 10x18 grid,
        # 640 -> 8x16, 768 -> 8x14. That's a SECOND, hidden compression happening
        # before FlashVID even runs, invisible in the reported retention (which
        # only reflects FlashVID's own ratio against the already-shrunk base).
        # Override longest_edge per-call so OUR chosen per-frame resolution
        # (already fixed by tools.py's smart_resize) is what actually gets used,
        # and FlashVID's retention_ratio is the only compression in effect.
        T, _, H, W = video_tensor.shape
        size_override = {"shortest_edge": 4096, "longest_edge": T * H * W * 2}
        t0 = time.time()
        vp = self.processor.video_processor(
            videos=[video_tensor], do_sample_frames=False, size=size_override,
            return_tensors="pt"
        )
        t_proc = time.time() - t0
        pixel_values = vp["pixel_values_videos"].to(self.device, torch.bfloat16)
        grid_thw = vp["video_grid_thw"].to(self.device)
        t, h, w = grid_thw[0].tolist()
        tpf = (h // 2) * (w // 2)
        base_tokens = t * tpf

        t0 = time.time()
        hidden, deepstack, cls_attention = self.model.model.visual(pixel_values, grid_thw)
        torch.cuda.synchronize(self.device)
        t_vit = time.time() - t0

        auto_r = retention_for_budget(base_tokens, target_tokens)
        if config.FIXED_RETENTION > 0:
            # On short spans, fixed retention would produce fewer tokens than
            # crop_video's budget and become a blurry crop. Keep the nominal
            # matched-budget ratio there; use calibrated fixed retention for
            # long spans where it preserves the intended coverage advantage.
            retention = (
                max(config.FIXED_RETENTION, auto_r)
                if T < config.FLOOR_ENGAGE_FRAMES
                else config.FIXED_RETENTION
            )
        else:
            retention = auto_r
        fv_kw = dict(config.FLASHVID_KW)
        fv_kw["retention_ratio"] = retention
        valid = {f.name for f in dataclasses.fields(FlashVidConfig)}
        fv_cfg = FlashVidConfig(**{k: v for k, v in fv_kw.items() if k in valid})
        fv_cfg.H, fv_cfg.W = h // 2, w // 2

        video_features = hidden.view(t, tpf, -1)
        # Small scalar/shape diagnostics only.  Keeping these in metadata makes
        # the compression boundary inspectable without retaining duplicate
        # feature tensors or changing FlashVID's inputs.
        diagnostics = {
            "input_video_tensor": list(video_tensor.shape),
            "processor_pixel_values": list(pixel_values.shape),
            "grid_thw": list(grid_thw.shape),
            "vision_hidden": list(hidden.shape),
            "video_features_before_flashvid": list(video_features.shape),
            "cls_attention_before_flashvid": list(cls_attention.shape),
            "deepstack_before_flashvid": [list(d.shape) for d in deepstack],
            "finite": {
                "pixel_values": bool(torch.isfinite(pixel_values).all().item()),
                "vision_hidden": bool(torch.isfinite(hidden).all().item()),
                "video_features": bool(torch.isfinite(video_features).all().item()),
                "cls_attention": bool(torch.isfinite(cls_attention).all().item()),
                "deepstack": all(bool(torch.isfinite(d).all().item()) for d in deepstack),
            },
        }
        t0 = time.time()
        compressed, keep_idx = flashvid_compression(
            video_features=video_features,
            cls_attention=cls_attention,
            flashvid_config=fv_cfg,
        )
        t_fv = time.time() - t0
        compressed = compressed.reshape(-1, hidden.shape[-1])
        keep_idx = keep_idx.to(self.device).long().reshape(-1)
        deepstack_kept = [d[keep_idx] for d in deepstack]  # anchors' rows (FlashVID line 324)

        # Native Qwen3-VL video layout (processing_qwen3_vl.py:218-227): per
        # temporal group `<{t:.1f} seconds><|vision_start|>pads<|vision_end|>`.
        # get_rope_index depends on this (splits the grid per-frame, t=1 each;
        # temporal position is carried by the timestamp TEXT).
        import numpy as np

        gt = np.asarray(frame_times, dtype=float)
        n_groups_have = len(gt) // 2
        group_times = gt[: n_groups_have * 2].reshape(-1, 2).mean(1)
        if len(group_times) < t:  # video processor may pad the last group
            pad = group_times[-1] if len(group_times) else 0.0
            group_times = np.concatenate([group_times, [pad] * (t - len(group_times))])
        seg = []
        for k in range(t):
            seg += self.tokenizer.encode(
                f"<{group_times[k]:.1f} seconds>", add_special_tokens=False
            )
            seg += [self.vision_start_id] + [self.video_token_id] * tpf + [self.vision_end_id]
        m = dict(meta or {})
        m.update(
            base_tokens=base_tokens,
            kept_tokens=int(compressed.shape[0]),
            retention=retention,
            grid=(t, h, w),
            t_proc=round(t_proc, 2),
            t_vit=round(t_vit, 2),
            t_flashvid=round(t_fv, 2),
            target_tokens=int(target_tokens),
            diagnostics=diagnostics,
        )
        return Clip(
            kind="video",
            segment_ids=torch.tensor(seg, dtype=torch.long, device=self.device),
            embeds=compressed,
            deepstack=deepstack_kept,
            grid_thw=grid_thw,
            keep_indices=keep_idx,
            meta=m,
        )
```


## Source snapshot: `fast_agent/agent_loop.py`

- Lines: `393`
- SHA-256: `771ea02fe1974e45113fa2f82349932806245029a78525cb0358560aa728337e`

```python
"""Our own decoding scheme: embedding-space assembly + manual greedy decode.

Every round re-prefills the full sequence from CACHED clip embeddings (each clip's
ViT encode happens exactly once, at tool-execution time). Compressed video clips
contribute only their kept-anchor positions: we lay out the FULL placeholder run,
compute M-RoPE over the full grid via get_rope_index (native), then gather the kept
positions — exactly FlashVID's own M-RoPE treatment (modeling_qwen3_vl.py:340-344),
without touching the text model.
"""

import json
import os
import re
import time
from dataclasses import dataclass

import torch
from transformers.cache_utils import DynamicCache

from . import config, data, tools
from . import trajectory as tj
from .model import Clip, Engine, clip_tokens


@dataclass
class Assembled:
    embeds: torch.Tensor          # (1, L, D)
    position_ids: torch.Tensor    # (3, 1, L)
    visual_mask: torch.Tensor     # (1, L) bool
    deepstack: list | None        # per-level (n_visual, D)
    next_pos: int                 # first decode position (M-RoPE text plane)
    ids: torch.Tensor             # (1, L) kept token ids (debug/parity)

IMG_PLACEHOLDER = "<|vision_start|><|image_pad|><|vision_end|>"
VID_PLACEHOLDER = "<|vision_start|><|video_pad|><|vision_end|>"

_TOOL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)
STOP_STRINGS = ("</tool_call>", "</answer>")


# ---------------------------------------------------------------------------
# messages: list of {"role": str, "parts": [str | Clip, ...]}

def _render_text(engine: Engine, messages: list, add_generation_prompt=True,
                 tools: list | None = None) -> tuple[str, list]:
    """Chat-template the conversation with literal placeholder strings standing in
    for clips. When `tools` is given, the model's native `# Tools` system block is
    injected (the hermes format it was trained on). Returns (text, ordered clips)."""
    clips, chat = [], []
    for m in messages:
        buf = []
        for p in m["parts"]:
            if isinstance(p, Clip):
                clips.append(p)
                if p.kind == "images":
                    buf.append(IMG_PLACEHOLDER * p.grid_thw.shape[0])
                else:
                    buf.append(VID_PLACEHOLDER)
            else:
                buf.append(p)
        chat.append({"role": m["role"], "content": "".join(buf)})
    kw = {"tokenize": False, "add_generation_prompt": add_generation_prompt}
    if tools:
        kw["tools"] = tools
    text = engine.processor.apply_chat_template(chat, **kw)
    return text, clips


def assemble(engine: Engine, messages: list, tools: list | None = None) -> Assembled:
    """Build (inputs_embeds, position_ids, visual_pos_masks, deepstack, next_pos)
    for one prefill. Token layout is identical to the native processor's for image
    clips (asserted in smoke); video clips use the full-grid layout then keep only
    FlashVID anchors."""
    dev = engine.device
    text, clips = _render_text(engine, messages, tools=tools)
    ids = engine.tokenizer(text, return_tensors="pt").input_ids[0].to(dev)

    # Expand singleton pad ids to full runs (per clip, in order).
    img_clips = [c for c in clips if c.kind == "images"]
    vid_clips = [c for c in clips if c.kind == "video"]
    out, img_i, img_frame_i, vid_i = [], 0, 0, 0
    skip_next_ve = False
    for tok in ids.tolist():
        if skip_next_ve:  # drop the <|vision_end|> that closed the video sentinel
            assert tok == engine.vision_end_id
            skip_next_ve = False
            continue
        if tok == engine.image_token_id:
            c = img_clips[img_i]
            
            t, h, w = c.grid_thw[img_frame_i].tolist()
            out += [tok] * (t * (h // 2) * (w // 2))
            img_frame_i += 1
            if img_frame_i == c.grid_thw.shape[0]:
                img_i, img_frame_i = img_i + 1, 0
        elif tok == engine.video_token_id:
            # Replace the whole <vs><|video_pad|><ve> sentinel with the clip's
            # native per-frame segment (timestamps + per-frame <vs>pads<ve> —
            # required by get_rope_index's per-frame grid split).
            assert out and out[-1] == engine.vision_start_id
            out.pop()
            out += vid_clips[vid_i].segment_ids.tolist()
            vid_i += 1
            skip_next_ve = True
        else:
            out.append(tok)
    full_ids = torch.tensor(out, dtype=torch.long, device=dev).unsqueeze(0)
    L = full_ids.shape[1]

    # M-RoPE over the FULL layout (native helper), then prune non-kept video pads.
    img_grids = torch.cat([c.grid_thw for c in img_clips]) if img_clips else None
    vid_grids = torch.cat([c.grid_thw for c in vid_clips]) if vid_clips else None
    position_ids, _ = engine.model.model.get_rope_index(
        full_ids, img_grids, vid_grids, attention_mask=torch.ones_like(full_ids)
    )

    embeds = engine.model.get_input_embeddings()(full_ids)
    visual_mask = torch.zeros(L, dtype=torch.bool, device=dev)

    img_pos = (full_ids[0] == engine.image_token_id).nonzero(as_tuple=True)[0]
    off = 0
    for c in img_clips:
        pos = img_pos[off : off + c.n_tokens]
        off += c.n_tokens
        embeds[0, pos] = c.embeds.to(embeds.dtype)
        visual_mask[pos] = True

    # Video clips: FlashVID keep_indices may contain DUPLICATE anchors (an ADTS
    # pick and a merge-tree anchor can share a position). Mirror FlashVID's own
    # semantics (modeling_qwen3_vl.py:325-344): scatter (last write wins), then
    # keep an INDEX LIST with duplicates — never a boolean mask, which dedupes
    # and desyncs from the deepstack row count.
    vid_pos = (full_ids[0] == engine.video_token_id).nonzero(as_tuple=True)[0]
    is_vid_pad = torch.zeros(L, dtype=torch.bool, device=dev)
    is_vid_pad[vid_pos] = True
    kept_video = []
    off = 0
    for c in vid_clips:
        pos = vid_pos[off : off + c.meta["base_tokens"]]
        off += c.meta["base_tokens"]
        kept = pos[c.keep_indices]              # duplicates preserved
        embeds[0, kept] = c.embeds.to(embeds.dtype)
        visual_mask[kept] = True
        kept_video.append(kept)

    non_video = torch.arange(L, device=dev)[~is_vid_pad]
    final_index = torch.sort(
        torch.cat([non_video] + kept_video)
    ).values if kept_video else torch.arange(L, device=dev)

    embeds = embeds[:, final_index]
    position_ids = position_ids[..., final_index]
    visual_final = visual_mask[final_index].unsqueeze(0)
    deepstack = None
    if clips:
        n_levels = len(clips[0].deepstack)
        deepstack = [
            torch.cat([c.deepstack[lv] for c in clips]).to(embeds.dtype)
            for lv in range(n_levels)
        ]
    next_pos = int(position_ids.max()) + 1
    return Assembled(embeds, position_ids, visual_final, deepstack, next_pos,
                     full_ids[:, final_index])


@torch.inference_mode()
def decode(engine: Engine, a: Assembled,
           max_new_tokens: int = config.MAX_NEW_TOKENS) -> str:
    """Greedy decode with manual KV cache; stops on </tool_call>, </answer>, or EOS."""
    lm = engine.model.model.language_model
    lm_head = engine.model.lm_head
    embed_tokens = engine.model.get_input_embeddings()
    L = a.embeds.shape[1]
    next_pos = a.next_pos

    past = DynamicCache()
    out = lm(
        inputs_embeds=a.embeds,
        position_ids=a.position_ids,
        past_key_values=past,
        cache_position=torch.arange(L, device=engine.device),
        use_cache=True,
        visual_pos_masks=a.visual_mask,
        deepstack_visual_embeds=a.deepstack,
    )
    gen = []
    next_id = int(lm_head(out.last_hidden_state[:, -1:]).argmax(-1))
    for s in range(max_new_tokens):
        if next_id in engine.eos_ids:
            break
        gen.append(next_id)
        text = engine.tokenizer.decode(gen)
        if any(text.endswith(t) for t in STOP_STRINGS):
            break
        e = embed_tokens(torch.tensor([[next_id]], device=engine.device))
        pos = torch.full((3, 1, 1), next_pos + s, device=engine.device, dtype=torch.long)
        out = lm(
            inputs_embeds=e,
            position_ids=pos,
            past_key_values=past,
            cache_position=torch.tensor([L + s], device=engine.device),
            use_cache=True,
        )
        next_id = int(lm_head(out.last_hidden_state[:, -1:]).argmax(-1))
    return engine.tokenizer.decode(gen)


# ---------------------------------------------------------------------------
def parse_tool_call(text: str) -> dict | None:
    m = _TOOL_RE.search(text)
    if not m:
        return None
    try:
        d = json.loads(m.group(1))
        args = d.get("arguments") or d.get("parameters") or {}
        return {"name": d.get("name", ""), "args": args}
    except json.JSONDecodeError:
        return {"name": "__malformed__", "args": {}}


def crop_budget_tokens(engine: Engine) -> int:
    """Crop-tool token ceiling, derived not hardcoded: 128 frames at the actual
    processor grid. Uses a 224x224 probe frame."""
    from PIL import Image
    import numpy as np

    probe = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
    proc = engine.processor.image_processor(images=[probe], return_tensors="pt")
    per_frame = clip_tokens(proc["image_grid_thw"])
    return per_frame * config.CROP_MAX_FRAMES


def run_sample(engine: Engine, row: dict, tool_names: tuple = (),
               max_rounds: int = config.MAX_ROUNDS, verbose: bool = False,
               record_dir: str | None = None) -> dict:
    """Drive one sample through a LongVT-style loop: generate -> execute the tool if
    one was called, else stop (baseline is single-turn automatically). Then ONE
    bounded finalizer turn if no answer letter emerged. Dual-scored: pred_strict
    (pre-finalizer, LongVT-faithful) and pred_lenient (post-finalizer, headline).
    Writes a replayable trajectory + montages under record_dir when given."""
    qid = str(row["question_id"])
    media = os.path.join(record_dir, "media", qid) if record_dir else None

    t_sample = time.time()
    dur = data.video_duration(row["video_path"])
    pils = tools.initial_frames(row["video_path"])
    clip0 = engine.encode_images(pils, meta={"role": "initial"})
    initial_montage = (tj.save_montage(pils, os.path.join(media, "initial.png"))
                       if media else None)

    # Native tool schemas -> `# Tools` block, present in EVERY round of a tool arm.
    schemas = [config.TOOL_SCHEMAS[t] for t in tool_names] if tool_names else None

    prompt = (
        data.format_question(row)
        + "\n\n" + config.initial_view_text(dur, len(pils))
        + (config.tool_instructions(dur, tool_names) if tool_names else "")
        + "\n\n" + config.ANSWER_INSTR
    )
    messages = [{"role": "user", "parts": [clip0, prompt]}]
    per_frame = clip_tokens(clip0.grid_thw[:1])          # matched-budget target (crop=128 -> real compression)
    target_tokens = per_frame * config.CROP_MAX_FRAMES

    calls, texts, rounds = [], [], []
    n_exec = 0                                            # tool executions so far

    def gen(label) -> tuple[str, dict]:
        """One assemble+decode turn; appends assistant text, returns (text, rec)."""
        t0 = time.time()
        a = assemble(engine, messages, tools=schemas)
        t1 = time.time()
        text = decode(engine, a)
        t2 = time.time()
        texts.append(text)
        rec = {"round": label, "context_tokens": int(a.embeds.shape[1]),
               "gen_seconds": round(t2 - t0, 2),
               "timing": {"assemble_seconds": round(t1 - t0, 3),
                          "decode_seconds": round(t2 - t1, 3),
                          "round_seconds": round(t2 - t0, 3)},
               "thinking": text, "action": None, "tool_result": None}
        if verbose:
            print(f"  [R{label}] ({a.embeds.shape[1]} tok ctx) {text[:300]}")
        messages.append({"role": "assistant", "parts": [text]})
        return text, rec

    for rnd in range(max_rounds + 1):
        text, rec = gen(rnd)
        tc = parse_tool_call(text)
        has_answer = "<answer>" in text                  # answer wins over a same-turn tool call
        will_exec = (tc is not None and not has_answer
                     and tc["name"] in tool_names and n_exec < max_rounds)
        if not will_exec:
            rec["action"] = {"kind": "answer" if has_answer else "stop"}
            rounds.append(rec)
            break

        n_blocks = len(_TOOL_RE.findall(text))            # native hermes may emit several
        s, e, err = tools.clamp_span(
            tc["args"].get("start_time"), tc["args"].get("end_time"), dur
        )
        rec["action"] = {"kind": "tool_call", "name": tc["name"],
                         "start": s, "end": e, "error": err}
        if n_blocks > 1:
            rec["action"]["extra_tool_calls_dropped"] = n_blocks - 1
        calls.append({"name": tc["name"], "start": s, "end": e, "error": err})
        if err:
            rounds.append(rec)
            messages.append({"role": "user", "parts": [f"<tool_response>\n{err}\n</tool_response>"]})
            continue
        t_tool = time.time()  # decode + ViT encode (montage cost is negligible)
        if tc["name"] == "crop_video":
            frames = tools.crop_frames(row["video_path"], s, e)
            clip = engine.encode_images(frames, meta={"role": "crop", "span": (s, e)})
            note = (f"crop_video: {len(frames)} full-detail frames covering "
                    f"{s:.0f}s-{e:.0f}s (1 fps).")
            mont = (tj.save_montage(frames, os.path.join(media, f"r{rnd}_crop.png"))
                    if media else None)
            rec["tool_result"] = {"tool": "crop_video", "span": [s, e],
                                  "n_frames": len(frames), "montage": mont}
        else:  # compress_video
            vt, times = tools.compress_tensor(row["video_path"], s, e)
            clip = engine.encode_video_compressed(
                vt, target_tokens, times, meta={"role": "compress", "span": (s, e)}
            )
            note = (f"compress_video: compressed overview of {s:.0f}s-{e:.0f}s "
                    f"({vt.shape[0]} frames -> {clip.meta['kept_tokens']} tokens).")
            mont = (tj.save_montage(vt, os.path.join(media, f"r{rnd}_compress.png"))
                    if media else None)
            rec["tool_result"] = {"tool": "compress_video", "span": [s, e],
                                  "n_frames": int(vt.shape[0]),
                                  "kept_tokens": clip.meta["kept_tokens"],
                                  "base_tokens": clip.meta["base_tokens"],
                                  "retention": round(clip.meta["retention"], 3),
                                  "montage": mont}
            calls[-1].update({k: clip.meta[k] for k in
                              ("kept_tokens", "base_tokens", "retention")})
        rec["tool_seconds"] = round(time.time() - t_tool, 2)
        n_exec += 1
        rounds.append(rec)
        messages.append({"role": "user",
                         "parts": ["<tool_response>\n", clip,
                                   f"\n{note}{config.TOOL_RESULT_INSTR}\n</tool_response>"]})

    # Strict, LongVT-faithful score: whatever the model produced on its own.
    pred_strict = data.extract_answer("\n".join(texts))

    # One bounded finalizer turn (identical across arms) only if no letter surfaced.
    finalizer_used = False
    if pred_strict is None:
        finalizer_used = True
        messages.append({"role": "user", "parts": [
            "Reply with <answer>X</answer> where X is one of A, B, C, or D."]})
        _text, rec = gen("finalizer")
        rec["action"] = {"kind": "finalizer"}
        rounds.append(rec)

    pred_lenient = data.extract_answer("\n".join(texts))
    gold = row["answer"]
    seconds = round(time.time() - t_sample, 1)
    traj_path = None
    if record_dir:
        traj = {"question_id": qid, "task_type": row["task_type"],
                "question": row["question"], "options": list(row["options"]),
                "gold": gold, "pred": pred_lenient,
                "pred_strict": pred_strict, "pred_lenient": pred_lenient,
                "finalizer_used": finalizer_used,
                "correct": pred_lenient == gold, "correct_strict": pred_strict == gold,
                "videoID": row["videoID"], "video_path": row["video_path"],
                "duration": dur, "tools": list(tool_names), "seconds": seconds,
                "schema_version": 3,
                "timing": {"sample_seconds": seconds,
                           "round_seconds": round(sum(r["timing"]["round_seconds"] for r in rounds), 3),
                           "tool_seconds": round(sum(r.get("tool_seconds", 0) for r in rounds), 3)},
                "initial": {"n_frames": len(pils), "montage": initial_montage},
                "rounds": rounds}
        traj_path = tj.save_trajectory(traj, os.path.join(record_dir, "traj", f"{qid}.json"))

    return {
        "question_id": qid,
        "task_type": row["task_type"],
        "pred": pred_lenient,
        "pred_strict": pred_strict,
        "pred_lenient": pred_lenient,
        "finalizer_used": finalizer_used,
        "gold": gold,
        "correct": pred_lenient == gold,
        "correct_strict": pred_strict == gold,
        "rounds": len(texts),
        "tool_calls": calls,
        "duration": dur,
        "seconds": seconds,
        "traj_path": traj_path,
    }
```


## Source snapshot: `fast_agent/trajectory.py`

- Lines: `57`
- SHA-256: `9c742c39f91c0154022e1044a29ecd28a29f96f9e621715b86005a8c3360f82a`

```python
"""Trajectory recording: montages of what the model saw + structured per-round JSON.

A trajectory JSON captures everything needed to replay a sample by eye:
  question/options/gold/pred, the initial skim montage, then per round the
  thinking text, the action taken, and (for tool calls) a montage of the frames
  that tool fed back. viz.py renders these in a notebook.
"""

import json
import math
import os

import numpy as np
import torch
from PIL import Image


def _to_pils(items, max_n: int) -> list:
    """Accept a list of PIL frames OR a (T,C,H,W) uint8 tensor; return <=max_n
    uniformly-spaced PIL frames."""
    if isinstance(items, torch.Tensor):
        T = items.shape[0]
        idx = np.linspace(0, T - 1, min(max_n, T)).astype(int)
        return [
            Image.fromarray(items[i].permute(1, 2, 0).cpu().numpy().astype("uint8"))
            for i in idx
        ]
    items = list(items)
    if len(items) > max_n:
        idx = np.linspace(0, len(items) - 1, max_n).astype(int)
        items = [items[i] for i in idx]
    return items


def save_montage(items, path: str, max_thumbs: int = 48, cols: int = 8,
                 thumb_w: int = 160) -> str | None:
    """Tile frames into a single PNG grid (downsampled). Returns path or None."""
    pils = _to_pils(items, max_thumbs)
    if not pils:
        return None
    w0, h0 = pils[0].size
    th = max(1, int(thumb_w * h0 / w0))
    rows = math.ceil(len(pils) / cols)
    canvas = Image.new("RGB", (cols * thumb_w, rows * th), (18, 18, 18))
    for i, im in enumerate(pils):
        r, c = divmod(i, cols)
        canvas.paste(im.convert("RGB").resize((thumb_w, th)), (c * thumb_w, r * th))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    canvas.save(path)
    return path


def save_trajectory(traj: dict, path: str) -> str:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(traj, f, indent=1)
    return path
```


## Source snapshot: `fast_agent/debug_pipeline.py`

- Lines: `438`
- SHA-256: `f7401d35fb705248d41f39f9a97660a4598a45abde37cad04d98ea27fcd904bb`

```python
"""One-sample diagnostic harness for the four explicitly separated debug modes.

This is intentionally not an evaluation runner.  It never loads more than one
dataset row and writes a transparent, human-readable record under
``debug_runs/<qid>/<mode>``.
"""

import argparse
import json
import traceback
from pathlib import Path

import torch

from . import config, data, tools
from .agent_loop import _TOOL_RE, _render_text, assemble, decode, parse_tool_call
from .model import Engine, clip_tokens
from .trajectory import save_montage


MODES = (
    "baseline_single_turn",
    "forced_crop",
    "forced_compress",
    "autonomous_tools",
)


class Artifacts:
    def __init__(self, root: Path, row: dict, mode: str, start: float, end: float):
        self.path = root / str(row["question_id"]) / mode
        self.visuals = self.path / "visualizations"
        self.visuals.mkdir(parents=True, exist_ok=True)
        self.generations = []
        self.prompts = []
        self.trace = []
        self.timestamps = {}
        self.shapes = {}
        self.config = {
            "mode": mode,
            "question_id": str(row["question_id"]),
            "video_id": row["videoID"],
            "video_path": row["video_path"],
            "model_snapshot": config.MODEL_SNAPSHOT,
            "manual_time_range_seconds": [start, end],
            "initial_frames": config.INITIAL_FRAMES,
            "crop_max_frames": config.CROP_MAX_FRAMES,
            "compress_max_frames": config.COMPRESS_MAX_FRAMES,
            "fixed_retention": config.FIXED_RETENTION,
            "max_new_tokens": config.MAX_NEW_TOKENS,
        }

    def finish(self, result: dict):
        self._json("config.json", self.config)
        self._json("parsed_result.json", result)
        self._json("frame_timestamps.json", self.timestamps)
        self._json("tensor_shapes.json", self.shapes)
        self._text("prompt.txt", "\n\n".join(self.prompts))
        self._text("raw_generations.txt", "\n\n".join(
            f"=== generation {i + 1} ===\n{text}" for i, text in enumerate(self.generations)
        ))
        self._text("trace.md", "\n\n".join(self.trace) + "\n")

    def fail(self, exc: Exception, component: str):
        self.trace.append(
            f"## Failure\n\nThe run stopped in the **{component}** component: "
            f"`{type(exc).__name__}: {exc}`."
        )
        self.finish({
            "status": "failed",
            "failure_component": component,
            "error": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc(),
            "parsed_answer": data.extract_answer("\n".join(self.generations)),
        })

    def _json(self, name: str, value):
        with open(self.path / name, "w") as f:
            json.dump(value, f, indent=2, sort_keys=True)

    def _text(self, name: str, value: str):
        with open(self.path / name, "w") as f:
            f.write(value)


def _shape(x):
    return list(x.shape)


def _frame_info(frames) -> dict:
    first = frames[0]
    if torch.is_tensor(first):
        return {"count": len(frames), "resolution_hw": [int(first.shape[-2]), int(first.shape[-1])]}
    return {"count": len(frames), "resolution_hw": [first.height, first.width]}


def _answer_status(raw: str, parsed: str | None) -> dict:
    if parsed is not None:
        return {"status": "worked"}
    # The unchanged parser deliberately accepts only A-D.  Outputs such as
    # <answer>Unknown</answer> are therefore model behavior, not parser bugs.
    return {"status": "failed", "failure_component": "model behavior",
            "error": "no answer could be parsed from the model output"}


def _question_prompt(row, duration: float, n_initial: int, tool_names=()) -> str:
    return (
        data.format_question(row)
        + "\n\n" + config.initial_view_text(duration, n_initial)
        + (config.tool_instructions(duration, tool_names) if tool_names else "")
        + "\n\n" + config.ANSWER_INSTR
    )


def _record_generation(engine, messages, art: Artifacts, tools_schema=None, label="1"):
    rendered, _clips = _render_text(engine, messages, tools=tools_schema)
    art.prompts.append(f"=== model input for generation {label} ===\n{rendered}")
    assembled = assemble(engine, messages, tools=tools_schema)
    text = decode(engine, assembled)
    art.generations.append(text)
    art.shapes[f"generation_{label}"] = {
        "input_embeds": _shape(assembled.embeds),
        "position_ids_mrope": _shape(assembled.position_ids),
        "visual_mask": _shape(assembled.visual_mask),
        "kept_input_ids": _shape(assembled.ids),
        "deepstack": [_shape(x) for x in (assembled.deepstack or [])],
        "finite": {
            "input_embeds": bool(torch.isfinite(assembled.embeds).all().item()),
            "position_ids": bool(torch.isfinite(assembled.position_ids).all().item()),
        },
    }
    messages.append({"role": "assistant", "parts": [text]})
    return text


def _prepare_initial(engine, row, art: Artifacts):
    frames, timestamps = tools.initial_frames_with_timestamps(row["video_path"])
    art.timestamps["initial"] = timestamps
    art.shapes["initial_frames"] = _frame_info(frames)
    save_montage(frames, str(art.visuals / "initial.png"))
    return frames, engine.encode_images(frames, meta={"role": "initial"})


def run_baseline(engine, row, art: Artifacts, start: float, end: float):
    duration = data.video_duration(row["video_path"])
    frames, initial = _prepare_initial(engine, row, art)
    prompt = _question_prompt(row, duration, len(frames))
    messages = [{"role": "user", "parts": [initial, prompt]}]
    raw = _record_generation(engine, messages, art)
    parsed = data.extract_answer(raw)
    generations, tool_calls = len(art.generations), 0
    terminated_after_first = generations == 1
    assert generations == 1, f"baseline made {generations} generations"
    assert tool_calls == 0, f"baseline made {tool_calls} tool calls"
    assert terminated_after_first, "baseline did not terminate after generation one"
    art.trace.extend([
        "# Baseline single turn",
        "1. The harness decoded the initial whole-video view and saved `visualizations/initial.png`.",
        "2. It combined the unchanged question, initial visual input, and answer instruction.",
        "3. It performed exactly one model generation and parsed that output directly.",
        "4. It stopped. No tool loop and no finalizer are reachable in this mode.",
    ])
    return {
        **_answer_status(raw, parsed), "parsed_answer": parsed, "gold": row["answer"],
        "raw_output": raw, "generation_count": generations,
        "tool_call_count": tool_calls, "terminated_after_first_generation": True,
        "assertions": {"exactly_one_generation": True, "zero_tool_calls": True,
                       "terminated_after_first_generation": True},
    }


def run_forced_crop(engine, row, art: Artifacts, start: float, end: float):
    duration = data.video_duration(row["video_path"])
    initial_frames, initial = _prepare_initial(engine, row, art)
    crop, timestamps = tools.crop_frames_with_timestamps(row["video_path"], start, end)
    art.timestamps["forced_crop"] = timestamps
    art.shapes["crop_frames"] = _frame_info(crop)
    montage = art.visuals / "forced_crop.png"
    save_montage(crop, str(montage))
    crop_clip = engine.encode_images(crop, meta={"role": "forced_crop", "span": [start, end]})
    prompt = _question_prompt(row, duration, len(initial_frames))
    note = (f"\n\nForced diagnostic crop for {start:.1f}s-{end:.1f}s: "
            f"{len(crop)} full-detail frames. Use this visual evidence to answer.")
    messages = [{"role": "user", "parts": [initial, prompt, note, crop_clip]}]
    raw = _record_generation(engine, messages, art)
    parsed = data.extract_answer(raw)
    assert len(art.generations) == 1
    art.trace.extend([
        "# Forced crop",
        f"1. The harness manually decoded {len(crop)} frames from {start:.1f}s to {end:.1f}s. The model did not choose or call a tool.",
        "2. Exact planned timestamps are in `frame_timestamps.json`; frame count and resolution are in `tensor_shapes.json`.",
        "3. The crop montage is `visualizations/forced_crop.png`.",
        "4. `prompt.txt` is the exact rendered model input after inserting the forced crop.",
        "5. The harness generated once, saved the raw output, parsed it, and stopped.",
    ])
    return {**_answer_status(raw, parsed), "parsed_answer": parsed, "gold": row["answer"],
            "raw_output": raw, "generation_count": 1, "tool_call_count": 0,
            "forced_operation": {"name": "crop_video", "span": [start, end]},
            "frame_count": len(crop), "resolution_hw": art.shapes["crop_frames"]["resolution_hw"]}


def _visual_text_context(rendered: str, placeholder: str, window: int = 500):
    pos = rendered.rfind(placeholder)
    if pos < 0:
        return {"placeholder": placeholder, "found": False}
    return {"placeholder": placeholder, "found": True,
            "text_before": rendered[max(0, pos - window):pos],
            "text_after": rendered[pos + len(placeholder):pos + len(placeholder) + window]}


def run_forced_compress(engine, row, art: Artifacts, start: float, end: float):
    duration = data.video_duration(row["video_path"])
    initial_frames, initial = _prepare_initial(engine, row, art)
    video_tensor, timestamps = tools.compress_tensor(row["video_path"], start, end)
    art.timestamps["forced_compress"] = timestamps
    art.shapes["compression_input_frames"] = {
        "tensor": _shape(video_tensor), "count": int(video_tensor.shape[0]),
        "resolution_hw": [int(video_tensor.shape[-2]), int(video_tensor.shape[-1])],
    }
    save_montage(video_tensor, str(art.visuals / "sampled_source_frames.png"))
    per_frame = clip_tokens(initial.grid_thw[:1])
    target = per_frame * config.CROP_MAX_FRAMES
    clip = engine.encode_video_compressed(
        video_tensor, target, timestamps,
        meta={"role": "forced_compress", "span": [start, end]},
    )
    art.shapes["flashvid_boundary"] = clip.meta["diagnostics"]
    art.shapes["flashvid_tokens"] = {
        "before": clip.meta["base_tokens"], "after": clip.meta["kept_tokens"],
        "expected_budget": target, "retention": clip.meta["retention"],
        "keep_indices": _shape(clip.keep_indices),
    }
    prompt = _question_prompt(row, duration, len(initial_frames))
    note = (f"\n\nForced diagnostic compression for {start:.1f}s-{end:.1f}s: "
            f"{video_tensor.shape[0]} source frames became {clip.meta['kept_tokens']} visual tokens. "
            "Use this visual evidence to answer.")
    messages = [{"role": "user", "parts": [initial, prompt, note, clip]}]
    rendered, _ = _render_text(engine, messages)
    art.shapes["inserted_visual_text_context"] = _visual_text_context(
        rendered, "<|vision_start|><|video_pad|><|vision_end|>"
    )

    assertions = {
        "finite_tensors": all(clip.meta["diagnostics"]["finite"].values()),
        "valid_dimensions": video_tensor.ndim == 4 and clip.embeds.ndim == 2
                            and clip.keep_indices.ndim == 1,
        "nonzero_token_count": clip.meta["base_tokens"] > 0 and clip.meta["kept_tokens"] > 0,
        "expected_token_budget": clip.meta["kept_tokens"] <= target,
    }
    raw = _record_generation(engine, messages, art)
    parsed = data.extract_answer(raw)
    art.trace.extend([
        "# Forced compression",
        f"1. The harness manually decoded {video_tensor.shape[0]} source frames from {start:.1f}s to {end:.1f}s. The model did not choose or call a tool.",
        "2. It passed the recorded input tensor through the existing vision tower and unchanged FlashVID compressor.",
        f"3. FlashVID reduced {clip.meta['base_tokens']} visual tokens to {clip.meta['kept_tokens']} against a budget of {target}.",
        "4. Boundary tensor shapes, finite checks, kept indices, and M-RoPE shapes are in `tensor_shapes.json`.",
        "5. Sampled source frames are in `visualizations/sampled_source_frames.png`.",
        "6. `prompt.txt` and `inserted_visual_text_context` show the exact text around the inserted visual representation.",
        "7. The harness generated once, saved the raw output, parsed it, and stopped.",
    ])
    assertion_error = None
    try:
        assert assertions["finite_tensors"], "non-finite tensor at FlashVID boundary"
        assert assertions["valid_dimensions"], "invalid compression tensor dimensions"
        assert assertions["nonzero_token_count"], "zero visual token count"
        assert assertions["expected_token_budget"], (
            f"kept {clip.meta['kept_tokens']} tokens, above expected token budget {target}"
        )
    except AssertionError as exc:
        assertion_error = str(exc)
    status = (_answer_status(raw, parsed) if assertion_error is None else {
        "status": "failed", "failure_component": "compression path",
        "error": f"AssertionError: {assertion_error}",
    })
    return {**status, "parsed_answer": parsed, "gold": row["answer"],
            "raw_output": raw, "generation_count": 1, "tool_call_count": 0,
            "forced_operation": {"name": "compress_video", "span": [start, end]},
            "tokens_before": clip.meta["base_tokens"], "tokens_after": clip.meta["kept_tokens"],
            "expected_token_budget": target, "assertions": assertions}


def run_autonomous(engine, row, art: Artifacts, start: float, end: float):
    duration = data.video_duration(row["video_path"])
    initial_frames, initial = _prepare_initial(engine, row, art)
    tool_names = ("crop_video", "compress_video")
    schemas = [config.TOOL_SCHEMAS[name] for name in tool_names]
    prompt = _question_prompt(row, duration, len(initial_frames), tool_names)
    messages = [{"role": "user", "parts": [initial, prompt]}]
    calls, events, n_exec, stop = [], [], 0, None
    target = clip_tokens(initial.grid_thw[:1]) * config.CROP_MAX_FRAMES

    for rnd in range(config.MAX_ROUNDS + 1):
        raw = _record_generation(engine, messages, art, schemas, str(rnd + 1))
        tc = parse_tool_call(raw)
        has_answer = "<answer>" in raw
        event = {"generation": rnd + 1, "raw_output": raw, "parsed_answer": data.extract_answer(raw),
                 "parsed_tool_call": tc}
        will_exec = (tc is not None and not has_answer and tc["name"] in tool_names
                     and n_exec < config.MAX_ROUNDS)
        if not will_exec:
            stop = "answer" if has_answer else "no_executable_tool_call"
            event["stopping_reason"] = stop
            events.append(event)
            break
        s, e, err = tools.clamp_span(tc["args"].get("start_time"), tc["args"].get("end_time"), duration)
        call = {"name": tc["name"], "start": s, "end": e, "error": err,
                "raw_tool_blocks": len(_TOOL_RE.findall(raw))}
        event["tool_call"] = call
        calls.append(call)
        if err:
            result = {"error": err}
            messages.append({"role": "user", "parts": [f"<tool_response>\n{err}\n</tool_response>"]})
        elif tc["name"] == "crop_video":
            frames, times = tools.crop_frames_with_timestamps(row["video_path"], s, e)
            art.timestamps[f"tool_{n_exec + 1}_crop"] = times
            clip = engine.encode_images(frames, meta={"role": "crop", "span": [s, e]})
            montage = art.visuals / f"tool_{n_exec + 1}_crop.png"
            save_montage(frames, str(montage))
            result = {"frame_count": len(frames), "resolution": _frame_info(frames)["resolution_hw"],
                      "montage": str(montage)}
            messages.append({"role": "user", "parts": ["<tool_response>\n", clip,
                f"\ncrop_video: {len(frames)} full-detail frames covering {s:.0f}s-{e:.0f}s (1 fps).\n</tool_response>"]})
        else:
            vt, times = tools.compress_tensor(row["video_path"], s, e)
            art.timestamps[f"tool_{n_exec + 1}_compress"] = times
            clip = engine.encode_video_compressed(vt, target, times, meta={"role": "compress", "span": [s, e]})
            montage = art.visuals / f"tool_{n_exec + 1}_compress.png"
            save_montage(vt, str(montage))
            result = {"frame_count": int(vt.shape[0]), "resolution": [int(vt.shape[-2]), int(vt.shape[-1])],
                      "tokens_before": clip.meta["base_tokens"], "tokens_after": clip.meta["kept_tokens"],
                      "montage": str(montage)}
            messages.append({"role": "user", "parts": ["<tool_response>\n", clip,
                f"\ncompress_video: compressed overview of {s:.0f}s-{e:.0f}s "
                f"({vt.shape[0]} frames -> {clip.meta['kept_tokens']} tokens).\n</tool_response>"]})
        event["tool_result"] = result
        events.append(event)
        n_exec += 1

    parsed = data.extract_answer("\n".join(art.generations))
    if parsed is None:
        messages.append({"role": "user", "parts": [
            "Reply with <answer>X</answer> where X is one of A, B, C, or D."]})
        raw = _record_generation(engine, messages, art, schemas, "finalizer")
        parsed = data.extract_answer("\n".join(art.generations))
        events.append({"generation": "finalizer", "raw_output": raw,
                       "parsed_answer": data.extract_answer(raw), "stopping_reason": "bounded_finalizer"})
        stop = "bounded_finalizer"
    art.shapes["events"] = events
    art.trace.append("# Autonomous tools")
    for event in events:
        art.trace.append(
            f"## Generation {event['generation']}\n\n"
            f"Parsed tool call: `{event.get('parsed_tool_call')}`. "
            f"Tool result: `{event.get('tool_result')}`. "
            f"Stopping reason: `{event.get('stopping_reason')}`. "
            f"Parsed answer for this generation: `{event.get('parsed_answer')}`."
        )
    status = _answer_status("\n".join(art.generations), parsed)
    return {**status, "parsed_answer": parsed, "gold": row["answer"],
            "generation_count": len(art.generations), "tool_call_count": len(calls),
            "tool_calls": calls, "stopping_reason": stop, "events": events}


RUNNERS = {
    "baseline_single_turn": run_baseline,
    "forced_crop": run_forced_crop,
    "forced_compress": run_forced_compress,
    "autonomous_tools": run_autonomous,
}


def _classify_failure(exc: Exception, mode: str) -> str:
    text = f"{type(exc).__name__}: {exc}".lower()
    if "cuda" in text or "nvidia" in text or "model" in text:
        return "harness"
    if mode == "forced_compress":
        return "compression path"
    if mode == "forced_crop":
        return "crop path"
    if "decode" in text or "frame" in text or "video" in text:
        return "crop path" if mode == "forced_crop" else "compression path"
    if "flashvid" in text or "token budget" in text or "tensor" in text:
        return "compression path"
    if "tool" in text and mode == "autonomous_tools":
        return "tool protocol"
    return "harness"


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--qid", default=None, help="exact question_id; default is seed-0 sample 1")
    parser.add_argument("--start", type=float, default=0.0)
    parser.add_argument("--end", type=float, default=300.0)
    parser.add_argument("--output-root", default="debug_runs")
    parser.add_argument("--mode", action="append", choices=MODES,
                        help="run only this mode (repeatable); default runs all four")
    args = parser.parse_args()
    rows = data.load_long_split()
    row = next((r for r in rows if str(r["question_id"]) == str(args.qid)), None) if args.qid else data.load_long_split(n=1, seed=0)[0]
    if row is None:
        raise SystemExit(f"question_id not found: {args.qid}")
    duration = data.video_duration(row["video_path"])
    start, end, err = tools.clamp_span(args.start, args.end, duration)
    if err:
        raise SystemExit(err)
    root = Path(args.output_root).resolve()
    selected_modes = tuple(args.mode) if args.mode else MODES

    try:
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA is unavailable; the configured local model requires a CUDA device")
        engine = Engine()
    except Exception as exc:
        for mode in selected_modes:
            art = Artifacts(root, row, mode, start, end)
            art.fail(exc, "harness")
        raise

    summary = {}
    for mode in selected_modes:
        art = Artifacts(root, row, mode, start, end)
        try:
            result = RUNNERS[mode](engine, row, art, start, end)
            art.finish(result)
            summary[mode] = result["status"]
        except Exception as exc:
            component = _classify_failure(exc, mode)
            art.fail(exc, component)
            summary[mode] = f"failed: {component}: {type(exc).__name__}: {exc}"
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    print(json.dumps({"question_id": str(row["question_id"]), "output": str(root), "modes": summary}, indent=2))


if __name__ == "__main__":
    main()
```
